# Tail-Shape Analysis of Degraded-Image Noise

**Goal:** determine which parametric family best describes the noise/residual
distribution in the degraded (NoisyLR) semiconductor images, so the robust
loss function used later is justified by evidence, not guesswork.

Candidate families fit and compared (by AIC/BIC and visual QQ-fit):
- **Gaussian (Normal)** — thin tails, symmetric
- **Laplace** — heavier tails than Gaussian, symmetric, motivates an L1-style loss
- **Student-t** — heavy tails controlled by degrees-of-freedom `df`; `df` small = very heavy tails
- **Generalized Gaussian (GGD)** — flexible family; shape parameter `beta` < 2 = heavier-than-Gaussian tails, `beta`=2 recovers Gaussian, `beta`=1 recovers Laplace
- **2-component Gaussian Mixture** — checks whether the data is better explained as a mixture (e.g. a "clean" component + a "high-noise" component) rather than one heavy-tailed family

**Important data-pairing note:** the 400 test `NoisyLR` `.npy` files are *not*
paired with the 1281 train `GT` `.npy` files (different images, different
resolutions: 128x128 vs 256x256). So there is no direct per-pixel
`NoisyLR - GT` residual available from what's confirmed so far.



 ## Noise_Distribution_Analysis

In [ ]:
import os, glob
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

try:
    from sklearn.mixture import GaussianMixture
    HAVE_SKLEARN = True
except ImportError:
    HAVE_SKLEARN = False
    print("sklearn not found -- mixture-model comparison will be skipped.")
    print("Install with: pip install scikit-learn --break-system-packages")


In [ ]:

PROJECT_ROOT = Path("..").resolve()

TEST_NOISY_DIR = PROJECT_ROOT / "Test_NoisyLR" / "NoisyLR"

TRAIN_GT_DIR = PROJECT_ROOT / "train" / "train" / "GT"

TRAIN_NOISY_DIR = PROJECT_ROOT / "train" / "train" / "NoisyLR"

MAX_IMAGES = 400       
SUBSAMPLE_PER_IMAGE = 20000  
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [3]:
def list_npy(folder, max_files=None):
    paths = sorted(glob.glob(os.path.join(folder, "*.npy")))
    paths = [p for p in paths if "__MACOSX" not in p and not os.path.basename(p).startswith("._")]
    if max_files:
        paths = paths[:max_files]
    return paths

def load_npy(path):
    return np.load(path).astype(np.float64)

def sample_pixels(paths, n_per_image=SUBSAMPLE_PER_IMAGE):
    samples = []
    for p in paths:
        arr = load_npy(p).flatten()
        arr = arr[np.isfinite(arr)]
        if arr.size > n_per_image:
            idx = np.random.choice(arr.size, n_per_image, replace=False)
            arr = arr[idx]
        samples.append(arr)
    return np.concatenate(samples)

noisy_paths = list_npy(TEST_NOISY_DIR, MAX_IMAGES)
gt_paths = list_npy(TRAIN_GT_DIR, MAX_IMAGES)
print(f"NoisyLR files: {len(noisy_paths)}")
print(f"GT files: {len(gt_paths)}")

noisy_pixels = sample_pixels(noisy_paths)
print(f"Sampled {noisy_pixels.size} NoisyLR pixel values for the marginal analysis.")


NoisyLR files: 400
GT files: 400
Sampled 6553600 NoisyLR pixel values for the marginal analysis.


## Part 1 -- Fit candidate distributions to the raw NoisyLR pixel-value distribution

This is the primary, always-available analysis. It fits each family directly
to the observed degraded pixel values (not a residual, since no clean pair
exists at this resolution/dataset split) -- the tail *shape* is still fully
informative for choosing a robust loss, since the earlier Q1 diagnostic
already established GT lives in exactly [0,1] and NoisyLR spreads
continuously beyond it on both sides.

In [4]:
def fit_and_score(data, dist_name):
    """Fit a scipy.stats distribution by MLE and return (params, negLL, AIC, BIC, k_params)."""
    dist = getattr(stats, dist_name)
    params = dist.fit(data)
    logpdf = dist.logpdf(data, *params)
    negLL = -np.sum(logpdf)
    k = len(params)
    n = len(data)
    aic = 2 * k + 2 * negLL
    bic = k * np.log(n) + 2 * negLL
    return params, negLL, aic, bic, k

candidates = ["norm", "laplace", "t", "gennorm"]
results = {}
for name in candidates:
    params, negLL, aic, bic, k = fit_and_score(noisy_pixels, name)
    results[name] = {"params": params, "negLL": negLL, "AIC": aic, "BIC": bic, "k": k}
    print(f"{name:10s}  params={np.round(params,4)}  AIC={aic:,.1f}  BIC={bic:,.1f}")


norm        params=[0.4427 0.2843]  AIC=2,111,631.6  BIC=2,111,659.0
laplace     params=[0.4134 0.2368]  AIC=3,309,318.3  BIC=3,309,345.7
t           params=[8.23146621e+09 4.42700000e-01 2.84300000e-01]  AIC=2,111,633.6  BIC=2,111,674.7
gennorm     params=[3.3619 0.4705 0.4761]  AIC=1,843,888.9  BIC=1,843,930.0


In [5]:
# Optional: 2-component Gaussian Mixture comparison
if HAVE_SKLEARN:
    gmm = GaussianMixture(n_components=2, random_state=RANDOM_SEED, n_init=3)
    gmm.fit(noisy_pixels.reshape(-1, 1))
    gmm_negLL = -gmm.score(noisy_pixels.reshape(-1, 1)) * len(noisy_pixels)
    k_gmm = 2 * 3 - 1  # 2 means + 2 vars + 1 free weight
    n = len(noisy_pixels)
    gmm_aic = 2 * k_gmm + 2 * gmm_negLL
    gmm_bic = k_gmm * np.log(n) + 2 * gmm_negLL
    results["gmm_2comp"] = {"params": (gmm.means_.ravel(), gmm.covariances_.ravel(), gmm.weights_),
                             "negLL": gmm_negLL, "AIC": gmm_aic, "BIC": gmm_bic, "k": k_gmm}
    print(f"gmm_2comp  means={gmm.means_.ravel()}  weights={gmm.weights_}  AIC={gmm_aic:,.1f}  BIC={gmm_bic:,.1f}")


gmm_2comp  means=[0.24174306 0.68716227]  weights=[0.54874203 0.45125797]  AIC=1,497,405.6  BIC=1,497,474.1


In [6]:
print(f"{'Distribution':<12} {'AIC':>14} {'BIC':>14}  (lower is better)")
print("-" * 44)
for name, r in sorted(results.items(), key=lambda kv: kv[1]['AIC']):
    print(f"{name:<12} {r['AIC']:>14,.1f} {r['BIC']:>14,.1f}")
best_by_aic = min(results, key=lambda k: results[k]['AIC'])
print(f"\nBest fit by AIC: {best_by_aic}")


Distribution            AIC            BIC  (lower is better)
--------------------------------------------
gmm_2comp       1,497,405.6    1,497,474.1
gennorm         1,843,888.9    1,843,930.0
norm            2,111,631.6    2,111,659.0
t               2,111,633.6    2,111,674.7
laplace         3,309,318.3    3,309,345.7

Best fit by AIC: gmm_2comp


### Visualization: histogram with fitted PDFs overlaid (log-scale y-axis)

Log-scale makes tail differences between families visible -- a Gaussian's
tail drops off much faster (looks like a downward parabola on log-scale)
than a Laplace/Student-t/GGD tail (which looks closer to a straight or
gently-curving line).

In [9]:
x = np.linspace(noisy_pixels.min(), noisy_pixels.max(), 2000)

plt.figure(figsize=(10, 6))
plt.hist(noisy_pixels, bins=300, density=True, alpha=0.4, color="gray", label="NoisyLR pixel values")

for name in candidates:
    dist = getattr(stats, name)
    params = results[name]["params"]
    plt.plot(x, dist.pdf(x, *params), label=f"{name} fit (AIC={results[name]['AIC']:.0f})", linewidth=2)

plt.yscale("log")
plt.xlabel("pixel value")
plt.ylabel("density (log scale)")
plt.title("NoisyLR pixel-value distribution vs. fitted parametric families")
plt.legend()
plt.tight_layout()
plt.savefig("tail_shape_histogram_fits.png", dpi=150)
plt.show()
print("Saved: tail_shape_histogram_fits.png")


Saved: tail_shape_histogram_fits.png


C:\Users\gk192\AppData\Local\Temp\ipykernel_22484\2351811135.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### QQ-plots against each fitted family

A QQ-plot that hugs the diagonal line means that family fits well,
*especially* check the tails (top-right and bottom-left corners of each
plot) -- systematic curvature there is the clearest visual sign of which
family's tail behavior actually matches the data.

In [10]:
fig, axes = plt.subplots(1, len(candidates), figsize=(5 * len(candidates), 5))
sample_for_qq = np.random.choice(noisy_pixels, min(20000, len(noisy_pixels)), replace=False)

for ax, name in zip(axes, candidates):
    dist = getattr(stats, name)
    params = results[name]["params"]
    stats.probplot(sample_for_qq, dist=dist, sparams=params, plot=ax)
    ax.set_title(f"QQ-plot vs {name}")

plt.tight_layout()
plt.savefig("tail_shape_qqplots.png", dpi=150)
plt.show()
print("Saved: tail_shape_qqplots.png")


Saved: tail_shape_qqplots.png


C:\Users\gk192\AppData\Local\Temp\ipykernel_22484\427730593.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Skewness / kurtosis and automated interpretation

In [11]:
skew = stats.skew(noisy_pixels)
kurt = stats.kurtosis(noisy_pixels, fisher=True)  # excess kurtosis; 0 = Gaussian-like, >0 = heavy-tailed

print(f"Skewness (0 = symmetric):                 {skew:.4f}")
print(f"Excess kurtosis (0 = Gaussian, >0 heavy):  {kurt:.4f}")

t_params = results["t"]["params"]
gennorm_params = results["gennorm"]["params"]
# scipy.stats.t params: (df, loc, scale); scipy.stats.gennorm params: (beta, loc, scale)
t_df = t_params[0]
ggd_beta = gennorm_params[0]

print(f"\nFitted Student-t degrees of freedom (df): {t_df:.3f}  (smaller df = heavier tails; df>30 ~ Gaussian-like)")
print(f"Fitted Generalized-Gaussian beta:          {ggd_beta:.3f}  (beta=2 -> Gaussian, beta=1 -> Laplace, beta<1 -> heavier than Laplace)")


Skewness (0 = symmetric):                 0.4123
Excess kurtosis (0 = Gaussian, >0 heavy):  -0.5764

Fitted Student-t degrees of freedom (df): 8231466212.282  (smaller df = heavier tails; df>30 ~ Gaussian-like)
Fitted Generalized-Gaussian beta:          3.362  (beta=2 -> Gaussian, beta=1 -> Laplace, beta<1 -> heavier than Laplace)


In [12]:
print("="*70)
print("AUTOMATED RECOMMENDATION")
print("="*70)

if best_by_aic == "norm":
    print("Best fit: Gaussian -> tails are NOT unusually heavy.")
    print("Recommendation: a standard L2 / Charbonnier loss is likely well justified;")
    print("robust down-weighting is probably unnecessary.")
elif best_by_aic in ("laplace",):
    print("Best fit: Laplace -> moderately heavy, symmetric tails.")
    print("Recommendation: an L1-style loss (or Charbonnier, which approximates L1")
    print("away from zero) is well justified; robust M-estimator losses (Huber) are")
    print("a reasonable middle ground.")
elif best_by_aic == "t":
    print(f"Best fit: Student-t (df={t_df:.2f}) -> heavy tails, degree set by df.")
    if t_df < 5:
        print("df is small -> VERY heavy tails -> a strongly robust / redescending")
        print("loss (e.g. Tukey biweight, Cauchy loss) is justified: large residuals")
        print("should be substantially down-weighted, not just linearly (L1) weighted.")
    else:
        print("df is moderate -> mildly heavy tails -> Huber/Charbonnier robust loss")
        print("is a reasonable, simpler choice than a fully redescending estimator.")
elif best_by_aic == "gennorm":
    print(f"Best fit: Generalized Gaussian (beta={ggd_beta:.2f}).")
    if ggd_beta < 1:
        print("beta < 1 -> heavier than Laplace -> favor a strongly robust/redescending loss.")
    elif ggd_beta < 2:
        print("1 <= beta < 2 -> between Laplace and Gaussian -> Charbonnier/Huber is well justified.")
    else:
        print("beta ~ 2 -> close to Gaussian -> standard L2 is reasonable.")
elif best_by_aic == "gmm_2comp":
    print("Best fit: 2-component Gaussian mixture -> the data looks less like ONE")
    print("heavy-tailed family and more like a MIXTURE of a 'low-noise' regime and")
    print("a 'high-noise' regime (e.g. varying local speckle intensity across pixels).")
    print("Recommendation: consider a per-pixel noise-level-adaptive loss (e.g.")
    print("heteroscedastic weighting keyed to local variance) rather than a single")
    print("fixed robust loss shape.")

print()
print(f"Skew={skew:.3f}, excess kurtosis={kurt:.3f} -- ", end="")
if abs(skew) > 0.3:
    print("noticeable asymmetry: consider an ASYMMETRIC robust loss")
    print("(e.g. asymmetric Huber / quantile-style loss) rather than a symmetric one.")
else:
    print("close to symmetric: a symmetric robust loss is adequate.")


AUTOMATED RECOMMENDATION
Best fit: 2-component Gaussian mixture -> the data looks less like ONE
heavy-tailed family and more like a MIXTURE of a 'low-noise' regime and
a 'high-noise' regime (e.g. varying local speckle intensity across pixels).
Recommendation: consider a per-pixel noise-level-adaptive loss (e.g.
heteroscedastic weighting keyed to local variance) rather than a single
fixed robust loss shape.

Skew=0.412, excess kurtosis=-0.576 -- noticeable asymmetry: consider an ASYMMETRIC robust loss
(e.g. asymmetric Huber / quantile-style loss) rather than a symmetric one.


## Part 2 (optional, stronger) -- true paired residual analysis

Only runs if `TRAIN_NOISY_DIR` is set above to a folder of train-side
degraded `.npy` files with filenames matching `TRAIN_GT_DIR`. This computes
the actual per-pixel residual `NoisyLR - downsampled(GT)` and repeats the
same fitting/plotting on the *true* noise residual, which is more rigorous
than the marginal-value analysis in Part 1.

In [13]:
def match_pairs(gt_dir, noisy_dir):
    gt_files = {os.path.basename(p): p for p in list_npy(gt_dir)}
    noisy_files = {os.path.basename(p): p for p in list_npy(noisy_dir)}
    common = sorted(set(gt_files) & set(noisy_files))
    return [(gt_files[k], noisy_files[k]) for k in common]

def downsample(img, out_hw):
    """Simple area-average downsampling to match NoisyLR resolution."""
    H, W = img.shape[:2]
    oh, ow = out_hw
    fh, fw = H // oh, W // ow
    if fh < 1 or fw < 1:
        raise ValueError("GT is smaller than target -- check resolutions.")
    trimmed = img[:oh*fh, :ow*fw]
    return trimmed.reshape(oh, fh, ow, fw).mean(axis=(1, 3))

if TRAIN_NOISY_DIR and os.path.isdir(TRAIN_NOISY_DIR):
    pairs = match_pairs(TRAIN_GT_DIR, TRAIN_NOISY_DIR)
    print(f"Found {len(pairs)} matched GT/NoisyLR pairs.")
    if MAX_IMAGES:
        pairs = pairs[:MAX_IMAGES]

    residual_samples = []
    for gt_path, noisy_path in pairs:
        gt_img = load_npy(gt_path)
        noisy_img = load_npy(noisy_path)
        gt_ds = downsample(gt_img, noisy_img.shape[:2])
        resid = (noisy_img - gt_ds).flatten()
        resid = resid[np.isfinite(resid)]
        if resid.size > SUBSAMPLE_PER_IMAGE:
            idx = np.random.choice(resid.size, SUBSAMPLE_PER_IMAGE, replace=False)
            resid = resid[idx]
        residual_samples.append(resid)

    residuals = np.concatenate(residual_samples)
    print(f"Sampled {residuals.size} true residual values.")

    resid_results = {}
    for name in candidates:
        params, negLL, aic, bic, k = fit_and_score(residuals, name)
        resid_results[name] = {"params": params, "negLL": negLL, "AIC": aic, "BIC": bic}
        print(f"{name:10s}  params={np.round(params,4)}  AIC={aic:,.1f}  BIC={bic:,.1f}")

    best_resid = min(resid_results, key=lambda k: resid_results[k]['AIC'])
    print(f"\nBest fit on TRUE RESIDUALS by AIC: {best_resid}")

    xr = np.linspace(residuals.min(), residuals.max(), 2000)
    plt.figure(figsize=(10, 6))
    plt.hist(residuals, bins=300, density=True, alpha=0.4, color="gray", label="true residual (NoisyLR - GT_downsampled)")
    for name in candidates:
        dist = getattr(stats, name)
        params = resid_results[name]["params"]
        plt.plot(xr, dist.pdf(xr, *params), label=f"{name} fit", linewidth=2)
    plt.yscale("log")
    plt.xlabel("residual value")
    plt.ylabel("density (log scale)")
    plt.title("True residual distribution vs. fitted parametric families")
    plt.legend()
    plt.tight_layout()
    plt.savefig("tail_shape_residual_fits.png", dpi=150)
    plt.show()
    print("Saved: tail_shape_residual_fits.png")
else:
    print("TRAIN_NOISY_DIR not set or not found -- skipping Part 2.")
    print("Part 1 (marginal NoisyLR distribution analysis) above is the result to use.")


TRAIN_NOISY_DIR not set or not found -- skipping Part 2.
Part 1 (marginal NoisyLR distribution analysis) above is the result to use.


## Summary

- Part 1 always gives a valid tail-shape verdict from the unpaired NoisyLR
  marginal distribution, since we already know (from the Q1 diagnostic) that
  GT occupies exactly [0,1] per image with no clipping -- so deviation shape
  in NoisyLR is informative on its own.
- Part 2, if you have a paired train-side NoisyLR folder, gives a stronger,
  directly-residual-based confirmation.
- Use the **AIC-best family** and the printed **automated recommendation**
  above to select the robust loss for the Phase 5 synthetic
  loss-comparison experiment (L1 vs. robust-loss), replacing the earlier
  TPG-C (censored) loss that Q1 ruled out.
